# Phase 2 — Profiling des données brutes — PanAfriPay

**Objectif** : prendre en main les 5 fichiers CSV livrés avec le sujet, comprendre leur structure, et confirmer concrètement les anomalies annoncées (doublons, MSISDN mal formatés, transactions miroir, vélocité anormale, dates de naissance aberrantes, intégrité référentielle) avant d'écrire le pipeline d'ingestion (Phase 3).

Ce notebook alimente directement le **rapport de profiling** attendu à l'issue de la Phase 2, et nourrit les choix d'architecture (règles de qualité, quarantaine) des phases suivantes.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

DATA_DIR = Path("../data/raw")

transactions = pd.read_csv(DATA_DIR / "transactions.csv")
customers = pd.read_csv(DATA_DIR / "customers.csv")
agents = pd.read_csv(DATA_DIR / "agents.csv")
operators = pd.read_csv(DATA_DIR / "operators.csv")
fx_rates = pd.read_csv(DATA_DIR / "fx_rates.csv")

print("Chargement terminé.")

Chargement terminé.


## 1. Vue d'ensemble des 5 fichiers

In [2]:
dfs = {
    "transactions": transactions,
    "customers": customers,
    "agents": agents,
    "operators": operators,
    "fx_rates": fx_rates,
}

for name, df in dfs.items():
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}")
    print(df.dtypes)
    print()

--- transactions ---
Shape: (109325, 18)
transaction_id          object
transaction_ref         object
transaction_type        object
sender_msisdn           object
receiver_msisdn         object
agent_id                object
amount                 float64
currency                object
fees                   float64
operator_code           object
country_code            object
transaction_status      object
initiated_at            object
completed_at            object
channel                 object
device_imei_hash        object
error_code              object
partner_merchant_id     object
dtype: object

--- customers ---
Shape: (50050, 11)
customer_id           object
msisdn                object
first_name            object
last_name             object
birth_date            object
kyc_level             object
registration_date     object
country_code          object
region                object
customer_status       object
account_balance      float64
dtype: object

--- agents ---


In [3]:
transactions.head()

,transaction_id,transaction_ref,transaction_type,sender_msisdn,receiver_msisdn,agent_id,amount,currency,fees,operator_code,country_code,transaction_status,initiated_at,completed_at,channel,device_imei_hash,error_code,partner_merchant_id
0,0c376f02-eab7-4452-b1ad-e43b2e6971bd,TRX-20260204-00094870,TRANSFER,+224737197273,+224765665858,NaN,2926.42,GNF,49.67,WAVE,GIN,PENDING,2026-02-04 18:39:17+00:00,NaN,APP,8a093b646e34fcb1,E_DECLINED,NaN
1,06b854dd-5566-42b6-9ef4-0b4c81143ef8,TRX-20260321-00038188,WITHDRAWAL,+225621567974,NaN,AGT-00103077,99483.15,XOF,1284.41,OM,CIV,SUCCESS,2026-03-21 17:08:21+00:00,2026-03-21 17:09:04+00:00,APP,e83fa295eafe9f2d,NaN,NaN
2,fbc47503-fc7a-4b8c-bae7-4aa38422b3a8,TRX-20260321-00021392,DEPOSIT,+224708965760,NaN,AGT-00103620,24844.35,GNF,200.17,OM,GIN,SUCCESS,2026-03-21 20:16:47+00:00,2026-03-21 20:17:06+00:00,USSD,NaN,NaN,NaN
3,804c2d11-c518-4df9-aa85-646f49b36a41,TRX-20260122-00008249,TRANSFER,+224780015245,+223630964801,NaN,6371.72,GNF,110.48,FREE,GIN,SUCCESS,2026-01-22 15:51:08+00:00,2026-01-22 15:51:12+00:00,APP,NaN,NaN,NaN
4,612a4b69-792f-4291-8f1a-ccac941f500e,TRX-20260224-00073838,BILL_PAYMENT,+223774790993,NaN,NaN,75553.56,XOF,570.54,FREE,MLI,SUCCESS,2026-02-24 07:52:47+00:00,2026-02-24 07:53:32+00:00,USSD,NaN,NaN,BILLER-681


## 2. Valeurs manquantes

Le sujet précise que certains champs peuvent être nuls légitimement (`receiver_msisdn` pour un retrait cash, `completed_at` si PENDING/FAILED, `device_imei_hash`, `partner_merchant_id`, `latitude`/`longitude` pour certains agents). L'objectif ici n'est pas de tout combler, mais de distinguer le null légitime du null anormal.

In [4]:
for name, df in dfs.items():
    print(f"--- {name} : % de valeurs manquantes par colonne ---")
    missing_pct = (df.isna().mean() * 100).round(2)
    print(missing_pct[missing_pct > 0].sort_values(ascending=False))
    print()

--- transactions : % de valeurs manquantes par colonne ---
error_code             88.11
partner_merchant_id    81.99
receiver_msisdn        69.89
agent_id               52.49
device_imei_hash       25.41
completed_at           11.89
dtype: float64

--- customers : % de valeurs manquantes par colonne ---
Series([], dtype: float64)

--- agents : % de valeurs manquantes par colonne ---
latitude     3.2
longitude    3.2
dtype: float64

--- operators : % de valeurs manquantes par colonne ---
Series([], dtype: float64)

--- fx_rates : % de valeurs manquantes par colonne ---
Series([], dtype: float64)



## 3. Doublons techniques

Anomalies attendues (sujet, section 10.3) :
- ~5% de doublons sur `transaction_id` dans `transactions.csv`
- 50 doublons de `customer_id` dans `customers.csv`
- 5 doublons d'`agent_id` dans `agents.csv`

In [5]:
dup_transactions = transactions[transactions.duplicated(subset="transaction_id", keep=False)]
dup_customers = customers[customers.duplicated(subset="customer_id", keep=False)]
dup_agents = agents[agents.duplicated(subset="agent_id", keep=False)]

n_tx = transactions["transaction_id"].duplicated().sum()
n_cust = customers["customer_id"].duplicated().sum()
n_agt = agents["agent_id"].duplicated().sum()

print(f"transactions : {n_tx} transaction_id dupliqués ({n_tx / len(transactions) * 100:.2f}% du fichier)")
print(f"customers    : {n_cust} customer_id dupliqués (attendu : 50)")
print(f"agents       : {n_agt} agent_id dupliqués (attendu : 5)")

transactions : 5000 transaction_id dupliqués (4.57% du fichier)
customers    : 50 customer_id dupliqués (attendu : 50)
agents       : 5 agent_id dupliqués (attendu : 5)


In [6]:
# Vérifier si les doublons sont des lignes identiques (dup technique pure) ou des conflits (mêmes ID, valeurs différentes)
exact_dup_tx = transactions.duplicated(keep=False).sum()
print(f"Lignes 100% identiques (transactions) : {exact_dup_tx}")
print(f"Doublons sur transaction_id mais lignes différentes (conflits) : {n_tx * 2 - exact_dup_tx if n_tx * 2 > exact_dup_tx else 0}")
dup_transactions.sort_values("transaction_id").head(10)

Lignes 100% identiques (transactions) : 9860
Doublons sur transaction_id mais lignes différentes (conflits) : 140


,transaction_id,transaction_ref,transaction_type,sender_msisdn,receiver_msisdn,agent_id,amount,currency,fees,operator_code,country_code,transaction_status,initiated_at,completed_at,channel,device_imei_hash,error_code,partner_merchant_id
103122,0005d037-70f8-40c0-a960-1dd845f468b4,TRX-20260208-00004677,TRANSFER,+221738648715,+225770151203,NaN,10760.61,XOF,92.43,FREE,SEN,SUCCESS,2026-02-08 10:13:35+00:00,2026-02-08 10:14:06+00:00,APP,3267c20dd4eb1b51,NaN,NaN
26729,0005d037-70f8-40c0-a960-1dd845f468b4,TRX-20260208-00004677,TRANSFER,+221738648715,+225770151203,NaN,10760.61,XOF,92.43,FREE,SEN,SUCCESS,2026-02-08 10:13:35+00:00,2026-02-08 10:14:06+00:00,APP,3267c20dd4eb1b51,NaN,NaN
43687,000686e4-3d85-4ca3-985e-41e8f91daf1b,TRX-20260324-00045602,DEPOSIT,+221751902258,NaN,AGT-00104700,145252.93,XOF,2287.50,FREE,SEN,SUCCESS,2026-03-24 20:34:48+00:00,2026-03-24 20:35:29+00:00,USSD,c49f3ef376853444,NaN,NaN
8712,000686e4-3d85-4ca3-985e-41e8f91daf1b,TRX-20260324-00045602,DEPOSIT,+221751902258,NaN,AGT-00104700,145252.93,XOF,2287.50,FREE,SEN,SUCCESS,2026-03-24 20:34:48+00:00,2026-03-24 20:35:29+00:00,USSD,c49f3ef376853444,NaN,NaN
48149,00192b76-58a1-47cb-b298-fca31b741308,TRX-20260301-00075272,TRANSFER,+223641492806,+221602915684,NaN,93866.08,XOF,1361.15,MTN_MOMO,MLI,SUCCESS,2026-03-01 09:20:43+00:00,2026-03-01 09:21:25+00:00,USSD,NaN,NaN,NaN
49787,00192b76-58a1-47cb-b298-fca31b741308,TRX-20260301-00075272,TRANSFER,+223641492806,+221602915684,NaN,93866.08,XOF,1361.15,MTN_MOMO,MLI,SUCCESS,2026-03-01 09:20:43+00:00,2026-03-01 09:21:25+00:00,USSD,NaN,NaN,NaN
24880,0019c1ab-0fd9-432e-a78e-34ade6f6e3ac,TRX-20260209-00038722,BILL_PAYMENT,+225723182428,NaN,NaN,43322.67,XOF,687.59,FREE,CIV,SUCCESS,2026-02-09 18:56:28+00:00,2026-02-09 18:56:55+00:00,AGENT,3e707a7405206455,NaN,BILLER-467
37251,0019c1ab-0fd9-432e-a78e-34ade6f6e3ac,TRX-20260209-00038722,BILL_PAYMENT,+225723182428,NaN,NaN,43322.67,XOF,687.59,FREE,CIV,SUCCESS,2026-02-09 18:56:28+00:00,2026-02-09 18:56:55+00:00,AGENT,3e707a7405206455,NaN,BILLER-467
2077,0035b850-5ead-4060-8030-b8c354d50148,TRX-20260103-00077888,MERCHANT_PAYMENT,+224765494198,NaN,NaN,35116.26,GNF,521.25,MOOV,GIN,SUCCESS,2026-01-03 08:50:36+00:00,2026-01-03 08:51:08+00:00,APP,64e12a711127f103,NaN,MCH-74972
96132,0035b850-5ead-4060-8030-b8c354d50148,TRX-20260103-00077888,MERCHANT_PAYMENT,+224765494198,NaN,NaN,35116.26,GNF,521.25,MOOV,GIN,SUCCESS,2026-01-03 08:50:36+00:00,2026-01-03 08:51:08+00:00,APP,64e12a711127f103,NaN,MCH-74972


## 4. Format des numéros MSISDN

Anomalie attendue : ~2,5% de MSISDN dans des formats inconsistants (avec/sans `+`, espaces, préfixe `00` au lieu de `+`). Format cible E.164 : `+221XXXXXXXXX`.

In [7]:
import re

E164_PATTERN = re.compile(r"^\+\d{10,15}$")

def classify_msisdn(value):
    if pd.isna(value):
        return "NULL"
    value = str(value).strip()
    if E164_PATTERN.match(value):
        return "E164_VALID"
    if value.startswith("00"):
        return "PREFIX_00"
    if " " in value:
        return "CONTAINS_SPACE"
    if value.startswith("+") is False and value.isdigit():
        return "MISSING_PLUS"
    return "OTHER_INVALID"

for col in ["sender_msisdn", "receiver_msisdn"]:
    print(f"--- {col} ---")
    counts = transactions[col].apply(classify_msisdn).value_counts()
    print(counts)
    print(f"Taux d'anomalie (hors NULL) : {counts.drop('NULL', errors='ignore').sum() / transactions[col].notna().sum() * 100:.2f}%")
    print()

print("--- customers.msisdn ---")
counts_cust = customers["msisdn"].apply(classify_msisdn).value_counts()
print(counts_cust)

--- sender_msisdn ---
sender_msisdn
E164_VALID        106750
PREFIX_00           1061
CONTAINS_SPACE      1017
MISSING_PLUS         497
Name: count, dtype: int64
Taux d'anomalie (hors NULL) : 100.00%

--- receiver_msisdn ---
receiver_msisdn
NULL              76403
E164_VALID        32176
PREFIX_00           293
CONTAINS_SPACE      293
MISSING_PLUS        160
Name: count, dtype: int64
Taux d'anomalie (hors NULL) : 100.00%

--- customers.msisdn ---
msisdn
E164_VALID        48848
CONTAINS_SPACE      491
PREFIX_00           470
MISSING_PLUS        241
Name: count, dtype: int64


## 5. Montants négatifs ou au-dessus du plafond réglementaire

Anomalie attendue : ~3% de transactions avec montant négatif ou supérieur au plafond quotidien de l'opérateur (`transaction_limit_daily` dans `operators.csv`).

In [8]:
negative_amounts = transactions[transactions["amount"] < 0]
print(f"Transactions à montant négatif : {len(negative_amounts)} ({len(negative_amounts) / len(transactions) * 100:.2f}%)")

# Jointure avec les plafonds opérateur
tx_with_limits = transactions.merge(
    operators[["operator_code", "transaction_limit_daily"]],
    on="operator_code",
    how="left"
)
over_limit = tx_with_limits[tx_with_limits["amount"] > tx_with_limits["transaction_limit_daily"]]
print(f"Transactions au-dessus du plafond quotidien : {len(over_limit)} ({len(over_limit) / len(transactions) * 100:.2f}%)")

total_anomaly_amount = len(negative_amounts) + len(over_limit)
print(f"Total anomalies de montant : {total_anomaly_amount} ({total_anomaly_amount / len(transactions) * 100:.2f}%) — attendu ~3%")

Transactions à montant négatif : 1508 (1.38%)
Transactions au-dessus du plafond quotidien : 1492 (1.36%)
Total anomalies de montant : 3000 (2.74%) — attendu ~3%


## 6. Intégrité référentielle : agent_id inexistant

Anomalie attendue : ~1% de transactions référençant un `agent_id` absent de `agents.csv`.

In [9]:
tx_with_agent = transactions[transactions["agent_id"].notna()]
orphan_agents = tx_with_agent[~tx_with_agent["agent_id"].isin(agents["agent_id"])]
print(f"Transactions avec agent_id qui n'existe pas dans agents.csv : {len(orphan_agents)}")
print(f"Soit {len(orphan_agents) / len(tx_with_agent) * 100:.2f}% des transactions liées à un agent (attendu ~1%)")
orphan_agents[["transaction_id", "agent_id", "transaction_type"]].head(10)

Transactions avec agent_id qui n'existe pas dans agents.csv : 1000
Soit 1.93% des transactions liées à un agent (attendu ~1%)


,transaction_id,agent_id,transaction_type
310,0cfca24e-27d6-40f0-83e3-9675eeec775e,AGT-9900388875,DEPOSIT
408,f78b8087-4770-445c-8b42-502b1e095aa0,AGT-9900782840,DEPOSIT
419,d83f132d-6c43-4b30-b993-b7081092e7ae,AGT-9900550911,DEPOSIT
565,95cf2b7a-271a-4b69-995f-68929a1918f2,AGT-9900292979,DEPOSIT
636,dd086289-088c-45e8-9cf7-a860e8dd9f3c,AGT-9900895317,DEPOSIT
654,4a61a177-d140-40d1-b5bf-1cd9ce2be5ff,AGT-9900892313,DEPOSIT
738,9397f192-3bc9-406e-afcf-dcd7524cc13e,AGT-9900817613,DEPOSIT
756,274a5fba-dc79-4e48-88c7-5fbcfd0a1243,AGT-9900191064,DEPOSIT
944,383d6896-baf4-40bb-bb45-066811f5fe72,AGT-9900267958,DEPOSIT
992,75f99cd9-4ccf-4dff-a269-7206a18f8a41,AGT-9900709538,DEPOSIT


## 7. Transactions miroir (indice de fraude)

Anomalie attendue : ~150 paires de transactions A→B et B→A pour le même montant exact, en moins de 60 secondes d'écart. Concerne les transactions de type `TRANSFER`.

In [10]:
transfers = transactions[transactions["transaction_type"] == "TRANSFER"].copy()
transfers["initiated_at"] = pd.to_datetime(transfers["initiated_at"], utc=True, errors="coerce")

# On cherche, pour chaque transfert A->B, un transfert B->A de même montant dans les 60s
merged = transfers.merge(
    transfers,
    left_on=["sender_msisdn", "receiver_msisdn", "amount"],
    right_on=["receiver_msisdn", "sender_msisdn", "amount"],
    suffixes=("_a", "_b")
)
merged["delta_seconds"] = (merged["initiated_at_b"] - merged["initiated_at_a"]).dt.total_seconds().abs()
mirror_candidates = merged[
    (merged["delta_seconds"] <= 60) &
    (merged["transaction_id_a"] != merged["transaction_id_b"])
]

n_mirror_pairs = len(mirror_candidates) // 2  # chaque paire apparaît 2 fois (A->B et B->A)
print(f"Paires de transactions miroir détectées : ~{n_mirror_pairs} (attendu ~150)")
mirror_candidates[["transaction_id_a", "transaction_id_b", "sender_msisdn_a", "receiver_msisdn_a", "amount", "delta_seconds"]].head(10)

Paires de transactions miroir détectées : ~150 (attendu ~150)


,transaction_id_a,transaction_id_b,sender_msisdn_a,receiver_msisdn_a,amount,delta_seconds
0,f9020669-cf50-426e-9e04-88813b1bb57d,2112a440-9079-43a0-bb99-e2f53c21dac7,+224604203398,+221641475482,151952.82,52.0
1,dc3e213b-514a-4d89-9184-ba552d764433,f4f5334b-012a-404e-895a-ad33ff6e7918,+221729229138,+221786880787,104707.70,29.0
2,01760cec-9b8c-41ad-8400-9dcf83545fbf,cfb45c06-f850-4aad-9a8c-299a5618cc7d,+225652165686,+221668377358,116592.88,39.0
3,0749f612-cb21-4406-894c-c1cbe4f1d11b,7b5b71eb-fa92-4502-b350-779137aa7238,+221754407195,+224760121724,112410.00,47.0
4,5d3182f0-5f1a-4993-a112-897ac6c6e0e1,1e314fbf-8382-48f5-b8aa-a83838e420ab,+224660697551,+224740756887,162920.20,26.0
5,0471bf75-d4de-4f4f-9dbf-74df154551b3,80cc80de-1658-4c45-9634-489698971cc5,+221652403498,+225725585416,111348.00,52.0
6,2b65dfdd-4103-420d-bed7-fa89fa008d5c,776dcf3d-1cd2-413d-b699-cb6392d22e1f,+225701355533,+221779125553,77944.65,22.0
7,1c7520e0-7d27-45f7-b040-a2cbc85cc24b,efb0702c-30b5-4897-b179-e1af5eff0c35,+224783638251,+221607746290,121911.52,54.0
8,b0dc95d6-7b35-40b4-8acd-d3273d0d7446,e30339a6-82de-41d5-978c-21daac7112ef,+221609968474,+221613306563,131432.56,18.0
9,0fa1aea0-8e89-4456-b47f-ea18dd4cc20b,f943d793-a19b-4873-a011-007f21bc4208,+223613594861,+223750849492,128154.95,35.0


## 8. Vélocité anormale

Anomalie attendue : au moins un client effectuant 25 transactions en 10 minutes.

In [11]:
tx_time = transactions.copy()
tx_time["initiated_at"] = pd.to_datetime(tx_time["initiated_at"], utc=True, errors="coerce")
tx_time = tx_time.sort_values(["sender_msisdn", "initiated_at"])

def max_count_in_window(group, window_minutes=10):
    times = group["initiated_at"].dropna().sort_values()
    if len(times) < 2:
        return 0
    counts = []
    for t in times:
        window_count = ((times >= t) & (times <= t + pd.Timedelta(minutes=window_minutes))).sum()
        counts.append(window_count)
    return max(counts) if counts else 0

velocity = tx_time.groupby("sender_msisdn").apply(max_count_in_window, include_groups=False)
suspicious = velocity[velocity >= 25].sort_values(ascending=False)
print(f"Clients avec >= 25 transactions en 10 minutes : {len(suspicious)}")
print(suspicious)

Clients avec >= 25 transactions en 10 minutes : 1
sender_msisdn
+225771702289    25
dtype: int64


## 9. Dates de naissance aberrantes (customers.csv)

Anomalie attendue : ~0,5% de dates de naissance manifestement erronées (`1900-01-01`, `9999-12-31`, `0000-00-00`, `1850-05-12`, etc.).

In [12]:
birth_dates_parsed = pd.to_datetime(customers["birth_date"], errors="coerce")
unparseable = customers[birth_dates_parsed.isna() & customers["birth_date"].notna()]

valid_range = birth_dates_parsed[birth_dates_parsed.notna()]
too_old = valid_range[valid_range.dt.year < 1920]
too_recent = valid_range[valid_range > pd.Timestamp.now()]

print(f"Dates non parsables (format invalide) : {len(unparseable)}")
print(f"Dates avant 1920 (probablement fausses) : {len(too_old)}")
print(f"Dates dans le futur : {len(too_recent)}")

total_bad_dates = len(unparseable) + len(too_old) + len(too_recent)
print(f"Total dates de naissance suspectes : {total_bad_dates} ({total_bad_dates / len(customers) * 100:.2f}%) — attendu ~0.5%")

customers.loc[too_old.index.union(unparseable.index), ["customer_id", "birth_date"]].head(10)

Dates non parsables (format invalide) : 134
Dates avant 1920 (probablement fausses) : 111
Dates dans le futur : 0
Total dates de naissance suspectes : 245 (0.49%) — attendu ~0.5%


,customer_id,birth_date
5,CUST-001000005,9999-12-31
251,CUST-001000251,9999-12-31
386,CUST-001000386,9999-12-31
793,CUST-001000793,0000-00-00
870,CUST-001000870,0000-00-00
1222,CUST-001001222,9999-12-31
1227,CUST-001001227,9999-12-31
1288,CUST-001001288,0000-00-00
1887,CUST-001001887,0000-00-00
1930,CUST-001001930,0000-00-00


## 10. Agents sans coordonnées géographiques

In [13]:
missing_coords = agents[agents["latitude"].isna() | agents["longitude"].isna()]
print(f"Agents sans latitude/longitude : {len(missing_coords)} ({len(missing_coords) / len(agents) * 100:.2f}%)")
missing_coords[["agent_id", "agent_name", "city", "latitude", "longitude"]].head(10)

Agents sans latitude/longitude : 160 (3.20%)


,agent_id,agent_name,city,latitude,longitude
103,AGT-00100103,Mini-marché Guédiawaye,Koulikoro,NaN,NaN
199,AGT-00100199,Kiosque Wakhinane,Kindia,NaN,NaN
234,AGT-00100234,Kiosque Touba,Dakar,NaN,NaN
237,AGT-00100237,Kiosque Sicap,Yamoussoukro,NaN,NaN
263,AGT-00100263,Marché Tabaski,Conakry,NaN,NaN
264,AGT-00100264,Boutique Liberté,San-Pédro,NaN,NaN
279,AGT-00100279,Boutique Ngor,Kaffrine,NaN,NaN
301,AGT-00100301,Marché Niary Tally,Nzérékoré,NaN,NaN
335,AGT-00100335,Cash & Go Ngor,Kaolack,NaN,NaN
348,AGT-00100348,Cash Point Plateau,Boké,NaN,NaN


## 11. Synthèse du profiling

À compléter après exécution — cette synthèse alimente directement :
- Le **rapport de profiling** (livrable Phase 2)
- Les **règles de qualité Great Expectations** (Phase 4)
- Les **règles de détection d'anomalies métier** (Phase 6, table `FACT_ANOMALY`)

| Anomalie | Attendu (sujet) | Constaté | Écart |
|---|---|---|---|
| Doublons transaction_id | ~5% | à compléter | |
| Montants négatifs/hors plafond | ~3% | à compléter | |
| MSISDN mal formatés | ~2.5% | à compléter | |
| agent_id orphelins | ~1% | à compléter | |
| Paires de transactions miroir | ~150 | à compléter | |
| Vélocité anormale (client) | ≥1 cas (25 tx/10min) | à compléter | |
| Dates de naissance aberrantes | ~0.5% | à compléter | |
| Doublons customer_id | 50 | à compléter | |
| Doublons agent_id | 5 | à compléter | |
| Agents sans coordonnées | non chiffré | à compléter | |
